# **Transform Races Data**
  1. Read bronze races table
  2. Keep only the columns required for analytics (Drop url column)
  3. Standardise Column using snake_case (raceName -> race_Name,circuitid -> circuit_id)
  4. Rename Columns to make them more meaningful ( date -> race_date)
  5. Remove duplicate records 
  6. Transform values of columns race_name to Title Case 
  7. Write transformated data to silver races table 







In [0]:
%run ../00-common/01_Environment-Config

In [0]:
bronze_table_nm = f"{catalog_name}.{bronze_schema}.races"
silver_table_nm = f"{catalog_name}.{silver_schema}.races"

### Step 1 - Read bronze races table

In [0]:
#Below is one way of reading data from table 
races_df = (
    spark.table(bronze_table_nm)    
)

In [0]:
#Below is another way of reading data from table this one allows to add options 
races_df = (
    spark.read.table(bronze_table_nm)   
    
)

In [0]:
display(races_df)

### Step 2 - Keep only the columns requried for analytics (Drop Url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
races_select_df =  races_df.select(
        F.col("season"),
        F.col("round"),        
        F.col("raceName"),
        F.col("date"),
        F.col("circuitId"),
        F.col("ingestion_timestamp"),
        F.col("source_file")
    )

    

### Step 3,4 - Standardise Column Names
- Standardize columns names using snake_case (raceName -> race_Name,circuitid -> circuit_id)
- Rename columns to make them more meaningful (date -> race_date)

In [0]:
races_renamed_df = (
    races_select_df.withColumnsRenamed(
        {
            "raceName": "race_Name",
            "date": "race_date",
            "circuitId": "circuit_Id"
        }
    )
)

In [0]:
display(races_renamed_df)

### Step -5 Remove Duplicates Records

In [0]:
 races_valid_df = races_renamed_df.dropDuplicates(["season", "round"])
 display(races_valid_df)

### Step - 6 Transform values of columns race_name to Title Case 

In [0]:
races_final_df = races_valid_df.withColumns({
    "race_Name": F.initcap(F.col("race_Name"))
})
display(races_final_df)

### Step - 7 Write transformated data to silver races table 

In [0]:
(
    races_final_df
        .write.mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table_nm)
)

In [0]:
spark.table(silver_table_nm).display()